# Projects Portal Issue

### `PIN 15681 Digital Delivery`

Projects Portal
- https://projects.udot.utah.gov/portal/home/index.html

Portal Item (Web Map)

- this is one of the Items with the issue
- https://projects.udot.utah.gov/portal/home/item.html?id=5f9786308dd24eb9aecf5e9a105e48c3
- Item ID: `5f9786308dd24eb9aecf5e9a105e48c3`
- Item Info page does not load in the Portal
- Service seems to be working, and its endpoint is available from the Server site

Server Service
- https://projects.udot.utah.gov/design/rest/services/PIN_15681_Digital_Delivery/FeatureServer

In [ ]:
from arcgis.gis import GIS
import pandas as pd
import helpers

# connect to GIS (Portal) using site admin creds
gis = helpers.get_gis('projects')

### Users

In [ ]:
derrick = "derricks@horrocks.com"
julia = "julia.downs@horrocks.com_project"
siteadmin = "gissiteadmin"

user_derrick = gis.users.get(derrick)
user_julia = gis.users.get(julia)
user_siteadmin = gis.users.get(siteadmin)

In [ ]:
user_derrick

In [ ]:
user_julia

## Original Item with the strange behavior
- does not load the Item Details page

NOTE: needs to be re-added to the group:  "15681 - Editing"


In [ ]:
item_id = "5f9786308dd24eb9aecf5e9a105e48c3"
item = gis.content.get(item_id)

item

### Example: Change ownership of Item using Python API

- Python API says it is possible to reassign the Item

In [ ]:
item.can_reassign(julia)

- Attempting to do that throws this exception:

`Exception: User folder does not exist.`

- not sure which User or which Folder

In [ ]:
item.reassign_to(target_owner=julia, target_folder=None)

## Unshare, reassign, reshare

* item that is inaccessible through the Portal site

In [ ]:
# item2_id = "31c699aeeb884f098a32040b228b5460"  # 15681 Digital Delivery Map Image Layer, can_reassign? false
item2_id = "2a070e10c70a467989a8798f679ab06e"  # Digital Construction Filter All Layers, can reassign? true

item2 = gis.content.get(item2_id)

item2

In [ ]:
item2.can_reassign(julia)

In [ ]:
# get the groups manually, if you messed up and deleted the list below

grp_ids = ["738902a80bd14e62be21fd7defd9ea9c", "1442efda2c31489e8a42c31c38daaa5a"]

item2_groups = []

for grp_id in grp_ids:
    item2_groups.append(gis.groups.get(grp_id))

item2_groups

In [ ]:
# store the groups that the item is shared with, for later undoing
shared_groups = []

In [ ]:
# get SharingManager object and add the Groups to the shared_groups list
sharing_mgr = item2.sharing

for group in sharing_mgr.shared_with['groups']:
    shared_groups.append(group)

for group in shared_groups:
    print(group)

In [ ]:
# remove the Item from all Groups using SharingGroupManager

# get SharingGroupManager object
item_grp_sharing_mgr = sharing_mgr.groups

for group in shared_groups:
    print(rf"{group.title} removed: {item_grp_sharing_mgr.remove(group)}")

In [ ]:
# show the Item's updated groups

# get SharingManager object
# sharing_mgr = item2.sharing

print(sharing_mgr.shared_with)

In [ ]:
# test if possible to reassign ownership
item2.can_reassign(julia)

In [ ]:
# reassign to Julia
item2.reassign_to(target_owner=julia, target_folder=None)

In [ ]:
# re-share Item with Groups

for group in shared_groups:
    print(rf"{group.title} re-shared: {item_grp_sharing_mgr.add(group)}")

In [ ]:
for group in sharing_mgr.shared_with['groups']:
    print(group)

### Special Groups

Earlier, we were seeing this exception:

`'message': 'Unable to reassign item. Item must not be shared to a special group.'`

In [ ]:
# list all Special Groups

special_groups = [grp for grp in gis.groups.search('') if 'updateitemcontrol' in grp.capabilities]

for grp in special_groups:
    print(f"{grp.title:<40} access: [{grp.access}]")

### Special Group Members

- the owner, Derrick, is a member of several Special Groups

In [ ]:
special_group_members = [(grp.title, grp.get_members()) for grp in gis.groups.search('') if ('updateitemcontrol' in grp.capabilities)]

for group in special_group_members:
    print(group[0])
    print(f"""    Owner : {group[1]['owner']}
    Admins: {', '.join(group[1]['admins'])}
    Users : {', '.join(group[1]['users'])}
    """)


Items owned by Derrick in Special Groups

In [ ]:
special_items = [grp.content() for grp in gis.groups.search('') if 'updateitemcontrol' in grp.capabilities]
items = list()

for item_list in special_items:
    if item_list:
        for item in item_list:
            if item.owner == derrick:
                items.append((item.title, item.id))

items_set = set(items)

for item in sorted(items_set):
    print(f"https://projects.udot.utah.gov/portal//home/item.html?id={item[1]}   {item[0]}")

### Check each Item's `.can_reassign()` results

In [ ]:
for item in sorted(items_set):
    # check if the item can be reassigned

    test_item = gis.content.get(item[1])
    # print(f"{test_item.title}")

    # check if the item can be reassigned to Julia
    results = test_item.can_reassign(julia)

    print(results)

### reassign the Items that tested True above

In [ ]:
for item in sorted(items_set):
    # check if the item can be reassigned

    test_item = gis.content.get(item[1])
    # print(f"{test_item.title}")

    # check if the item can be reassigned to Julia
    results = test_item.can_reassign(julia)

    if results[0]: # true
        try:
            test_item.reassign_to(julia)
        except Exception as err:
            print(err)


### Groups that these Items have been shared to

In [ ]:
item_group_member = []

for item in sorted(items_set):
    group_titles = []

    gis_item = gis.content.get(item[1])
    item_group_list = gis_item.sharing.groups.list()

    for group in item_group_list:
        group_titles.append(group.title)
    item_group_member.append(group_titles)

for groups in item_group_member:
    print(",  ".join(groups))

## Pandas dataframe


### Items

In [ ]:
item_list = []

item_ids = [
    '1d8305531f9d49ddbdcddea35b0a7c8c',
    '012c8c5a80974f09828f204ca8311e55',
    'f1f9578b4c7048ff82a10a8fab94aaed',
    '0c2929500756415e9447beb943ecce5b',
    '34e525459b674bde9c4bbab7dc87ba96',
    '31fad8ebba4440ff8c6d5bcf9dbc1572',
    '4e24bd045d1a49eda655ed146dcdad07',
    '5c6816b192d4458e81c5f7ec7d71d08a',
    '0eb9cc2858ef4e17bed338e7bfd4aaea',
    '060a6dd49d40404c9f3cf13fcf7a9a45',
    '1288861106644e17b44053b9df69cb10',
    'd179123a57b0449191d6507a4d465cd3',
    '0b56ab2e3f4545d199a6a4a7c516f645',
    'b11b40f1c71c4496b5545ef5f54958b9',
    'fdd2b7ebd6844bf1979d3c9bce820f07',
    'bfd1afcc159041bbaf120748d3e438c8',
    '60304bfd265a4b03b198e1a88cfce88c',
    '5f9786308dd24eb9aecf5e9a105e48c3',
    '31c699aeeb884f098a32040b228b5460',
    '58d9fbcda5e243f0ae3b0f3d3910775c',
    '2a070e10c70a467989a8798f679ab06e',
    'b54eafbf5078490f9631f9995f4e204e',
    'cf1b215789ce418591177eea5a7fe753',
    'f8c2d1dd458f4a7599f103df4e1e536b',
]

In [ ]:
for item_id in item_ids:

    item = gis.content.get(item_id)


    # reassign to Julia

    # check if the item can be reassigned to Julia
    can_julia = item.can_reassign(julia)


    item_dict = {
        "Title": item.title,
        "Item Type": item.type,
        "Owner": item.owner,
        "URL": rf"https://projects.udot.utah.gov/portal/home/item.html?id={item.id}",
        "ID": item.id,
        "can_reassign_julia": can_julia,
        }

    # attempt to reassign to Julia
    # add error message to dict, if unsuccessful

    if can_julia[0]: # true
        try:
            item.reassign_to(target_owner=julia, target_folder=None)
        except Exception as err:
            item_dict['reassign_msg'] = err

    # get all groups that this item is shared with
    item_group_list = item.sharing.groups.list()

    group_titles = []
    for group in item_group_list:
        group_titles.append(group.title)

    item_dict["Groups"] = "\n".join(group_titles)

    # append the info for this Item to the list
    item_list.append(item_dict)

In [ ]:
# create a dataframe from the list of Items

items_df = pd.DataFrame(item_list)
items_df = items_df.sort_values('ID')

items_df.info()

In [ ]:
items_df

In [ ]:
# copy the dataframe to the clipboard (paste into Google Sheets)
items_df.to_clipboard()

### Groups

In [ ]:
group_list = []

group_ids = [
    '332201fc9e97474483c7952395fabf48',
    '60a0575fd8d44af3b1cfc4a40e5040bb',
    '656b967d2c9f49e0b716a60da878bb8e',
    '738902a80bd14e62be21fd7defd9ea9c',
    'aafdfaaedfc84ff1bff07c470a9e9da8',
    'b351b6c1e69e4879b02e5b7284765793',
    'eed41e74ff9b4805a30d3228b91a5dd7',
]

for group_id in group_ids:

    group = gis.groups.get(group_id)

    members = group.get_members()

    admins = sorted(members['admins'])
    users = sorted(members['users'])


    group_dict = {
        "Title": group.title,
        "ID": group.id,
        "URL": rf"https://projects.udot.utah.gov/portal/home/group.html?id={group.id}",
        "Access": group.access,
        "Owner": members['owner'],
        "Admins": "\n".join(admins),
        "Users": "\n".join(users),
        }

    group_list.append(group_dict)

In [ ]:
groups_df = pd.DataFrame(group_list)

groups_df = groups_df.sort_values("Title")

groups_df.info()


In [ ]:
groups_df

In [ ]:
groups_df.to_clipboard()